<a href="https://colab.research.google.com/github/widura26/recap-automation/blob/main/recap_automation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Summary

Source : [link text](https://docs.google.com/spreadsheets/d/1Ibki18gicAFziEx1urTrvDv_KkzrkMMnRrbwqNYpOkw/edit?usp=sharing)

 List BOM
* Elektrik + Pre Series ✅ ✅
* De-Scoping ✅
* Interior ✅ ✅
* TMS ✅
* CKD VVVF ✅
* CKD Pantograph ✅
* CKD SIV ✅
* Elektrik ✅ ✅
* Komponen Utama ✅ ✅
* Sistem Mekanik ✅ ✅
* Raw Material Interior ✅
* Bogie KIT ✅
* Raw Material Bogie ✅
* Realisasi Raw Material ✅
* Carbody
* Fastening Mekanik
* Fastening Bogie
* Fastening Interior
* Crashwrothiness
* Welding
* Welding Bogie
* Consumable Tools ✅
* Consumable Series
* Welding WS BWI
* Jig Tools 6TS Pelokalan
* Jig Tool
* Tools ✅
* Tools & Consumable Tools Perbaikan ✅ ✅
* Consumable Pre Series ✅✅

---

Required data
*   Kode Material
*   Material (Nama material)
*   Spesifikasi
*   Qty / TS

In [8]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
import pandas as pd
import re

# Grabbing data (important)

In [170]:
#tambahan kolom : Stock total,

creds, _ = default()
client = gspread.authorize(creds)

# 1. Buka Spreadsheet
nama_file_sheets = "bom_dataset"
spreadsheet = client.open(nama_file_sheets)

# --- LIST LENGKAP ALL 29 TARGET SHEET ---
DAFTAR_SHEET_TARGET = [
    "Elektrik + Pre Series", "De-Scoping", "Interior", "TMS", "CKD VVVF",
    "CKD Pantograph", "CKD SIV", "Elektrik", "Komponen Utama", "Sistem Mekanik",
    "Raw Material Interior", "Bogie KIT", "Raw Material Bogie", "Realisasi Raw Material",
    "Carbody", "Fastening Mekanik", "Fastening Bogie", "Fastening Interior",
    "Crashwrothiness", "Welding", "Welding Bogie", "Consumable Tools",
    "Consumable Series", "Welding WS BWI", "Jig Tools 6TS Pelokalan", "jig Tool",
    "Tools", "Tools & Consumable Tools Perbaikan", "Consumable Pre Series"
]

semua_tab = spreadsheet.worksheets()
sheet_names_aktual = [sheet.title for sheet in semua_tab]

def normalize_text(text):
    """
    Normalisasi teks supaya perbedaan:
    - huruf besar/kecil
    - spasi
    - slash
    - underscore
    - tanda baca

    tidak terlalu berpengaruh.
    """
    if text is None:
        return ""

    text = str(text).strip().lower()

    # Hilangkan karakter non alphanumeric
    text = re.sub(r'[^a-z0-9]+', ' ', text)

    # Hilangkan spasi berlebih
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


def normalize_compact(text):
    """
    Contoh:
    'Kode Material'       -> 'kodematerial'
    'Kode-Material'       -> 'kodematerial'
    'kode_material'       -> 'kodematerial'
    """
    return re.sub(r'[^a-z0-9]', '', normalize_text(text))


def cari_sheet_aktual(nama_target):

    target = normalize_text(nama_target)
    target_compact = normalize_compact(nama_target)

    # Exact normalized
    for actual in sheet_names_aktual:
        if normalize_text(actual) == target:
            return actual

    # Compact
    for actual in sheet_names_aktual:
        if normalize_compact(actual) == target_compact:
            return actual

    return None


target_sheets_matched = []

for nama_target in DAFTAR_SHEET_TARGET:

    matched = cari_sheet_aktual(nama_target)

    if matched and matched not in target_sheets_matched:
        target_sheets_matched.append(matched)


print(
    f"[INFO] Memetakan "
    f"{len(target_sheets_matched)} dari "
    f"{len(DAFTAR_SHEET_TARGET)} target sheet.\n"
)


# ============================================================
# 4. MASTER HEADER
# ============================================================

KATA_KUNCI = {

    "kode": [
        "kode material",
        "kode",
        "material code",
        "material kode",
        "item code",
        "part number",
        "part no",
        "part no."
    ],

    "kode_stock": [
        "kode material stock",
        "kode material stok",
        "material stock code",
        "stock material code",
        "stock code",
        "kode stock",
        "kode stok"
    ],

    "nama": [
        "material / komponen",
        "material/komponen",
        "deskripsi material"
    ],

    "spek": [
        "spesifikasi",
        "specification",
        "spec",
        "detail spesifikasi",
        "SPESIFIKASI"
    ],

    "qty": [
        "qty / ts (series)",
        "qty/ts (series)",
        "qty ts series",
        "qty ts",
        "qty series",
        "quantity series"
    ],

    "uom": [
        "uom/car",
        "uom / car",
        "uom",
        "unit",
        "satuan"
    ],

    "qty_allowance": [
        "qty total (series)",
        # "qty total series",
        # "qty total",
        # "total qty",
        # "quantity total"
    ],

    "qty_pr_total": [
        "qty pr total",
        "qty pr",
        "pr total",
        "quantity pr total"
    ]
}


# ============================================================
# 5. DETEKSI HEADER DALAM SATU BARIS
# ============================================================

def cari_header_di_baris(row):

    hasil = {
        "kode": None,
        "kode_stock": None,
        "nama": None,
        "spek": None,
        "qty": None,
        "uom": None,
        "qty_allowance": None,
        "qty_pr_total": None
    }

    skor = 0

    for col_idx, cell in enumerate(row):

        cell_normalized = normalize_text(cell)
        cell_compact = normalize_compact(cell)

        if not cell_normalized:
            continue

        for field, keywords in KATA_KUNCI.items():

            # Jangan overwrite kalau sudah ketemu
            if hasil[field] is not None:
                continue

            for keyword in keywords:

                keyword_normalized = normalize_text(keyword)
                keyword_compact = normalize_compact(keyword)

                # Exact
                if cell_normalized == keyword_normalized:
                    hasil[field] = col_idx
                    skor += 3
                    break

                # Compact
                if cell_compact == keyword_compact:
                    hasil[field] = col_idx
                    skor += 3
                    break

                # Contains untuk header yang lebih panjang
                if (
                    len(keyword_normalized) >= 5
                    and keyword_normalized in cell_normalized
                ):
                    hasil[field] = col_idx
                    skor += 1
                    break

    return hasil, skor


# ============================================================
# 6. DETEKSI HEADER TERBAIK
# ============================================================

def deteksi_header(data, max_scan=100):

    kandidat = []

    jumlah_baris = min(max_scan, len(data))

    for row_idx in range(jumlah_baris):

        row = data[row_idx]

        hasil, skor = cari_header_di_baris(row)

        jumlah_field = sum(
            1 for value in hasil.values()
            if value is not None
        )

        # Minimal harus punya Kode atau Nama
        punya_kode = hasil["kode"] is not None
        punya_nama = hasil["nama"] is not None

        if punya_kode or punya_nama:

            kandidat.append({
                "row_idx": row_idx,
                "mapping": hasil,
                "score": skor,
                "jumlah_field": jumlah_field
            })

    if not kandidat:
        return None

    # Prioritaskan jumlah header yang ditemukan,
    # kemudian skor
    kandidat.sort(
        key=lambda x: (
            x["jumlah_field"],
            x["score"]
        ),
        reverse=True
    )

    return kandidat[0]


# ============================================================
# 7. EKSTRAKSI ANGKA
# ============================================================

def parse_number(value):

    if value is None:
        return 0.0

    text = str(value).strip()

    if not text:
        return 0.0

    text_upper = text.upper()

    invalid_values = [
        "N/A",
        "-",
        "#N/A",
        "#VALUE!",
        "#REF!",
        "#DIV/0!",
        "#NAME?",
        "#REF"
    ]

    if text_upper in invalid_values:
        return 0.0

    # Hilangkan pemisah ribuan sederhana
    text = text.replace(",", ".")

    match = re.search(
        r'-?\d+(?:\.\d+)?',
        text
    )

    if not match:
        return 0.0

    try:
        return float(match.group(0))
    except ValueError:
        return 0.0


# ============================================================
# 8. PROSES SEMUA SHEET
# ============================================================

rekap_data = []


for target_name in target_sheets_matched:

    sheet = spreadsheet.worksheet(target_name)

    data_mentah = sheet.get_all_values()

    if not data_mentah:

        print(
            f"Sheet '{target_name}' -> [SKIP] Tidak ada data."
        )

        continue


    # --------------------------------------------------------
    # DETEKSI HEADER
    # --------------------------------------------------------

    hasil_header = deteksi_header(
        data_mentah,
        max_scan=100
    )


    if hasil_header is None:

        print(
            f"Sheet '{target_name}' -> "
            f"[SKIP] Header tidak ditemukan."
        )

        continue


    header_idx = hasil_header["row_idx"]
    mapping = hasil_header["mapping"]

    print(
        f"\nSheet '{target_name}'"
    )

    print(
        f"  Header ditemukan pada baris "
        f"{header_idx + 1}"
    )

    print(
        f"  Jumlah header terdeteksi: "
        f"{hasil_header['jumlah_field']}"
    )

    print(
        f"  Mapping: {mapping}"
    )


    # --------------------------------------------------------
    # VALIDASI MINIMAL
    # --------------------------------------------------------

    if (
        mapping["kode"] is None
        and mapping["nama"] is None
    ):

        print(
            "  -> [SKIP] Tidak ada kolom Kode Material "
            "atau Material/Komponen."
        )

        continue


    # --------------------------------------------------------
    # EKSTRAKSI DATA
    # --------------------------------------------------------

    baris_terproses = 0


    for row in data_mentah[header_idx + 1:]:

        # Pastikan row cukup panjang
        max_index = max(
            [
                idx for idx in mapping.values()
                if idx is not None
            ],
            default=0
        )

        if len(row) <= max_index:
            row = row + [""] * (
                max_index + 1 - len(row)
            )


        # ----------------------------------------------------
        # AMBIL DATA
        # ----------------------------------------------------

        kode_material = (
            str(row[mapping["kode"]]).strip()
            if mapping["kode"] is not None
            else ""
        )

        kode_stock = (
            str(row[mapping["kode_stock"]]).strip()
            if mapping["kode_stock"] is not None
            else ""
        )

        nama_material = (
            str(row[mapping["nama"]]).strip()
            if mapping["nama"] is not None
            else ""
        )

        spesifikasi = (
            str(row[mapping["spek"]]).strip()
            if mapping["spek"] is not None
            else ""
        )

        uom_car = (
            str(row[mapping["uom"]]).strip()
            if mapping["uom"] is not None
            else ""
        )


        # ----------------------------------------------------
        # QTY
        # ----------------------------------------------------

        valid_qty = (
            parse_number(row[mapping["qty"]])
            if mapping["qty"] is not None
            else 0.0
        )


        # ----------------------------------------------------
        # ALLOWANCE
        # ----------------------------------------------------

        valid_allowance = (
            parse_number(
                row[mapping["qty_allowance"]]
            )
            if mapping["qty_allowance"] is not None
            else 0.0
        )


        # ----------------------------------------------------
        # PR TOTAL
        # ----------------------------------------------------

        valid_pr_total = (
            parse_number(
                row[mapping["qty_pr_total"]]
            )
            if mapping["qty_pr_total"] is not None
            else 0.0
        )


        # ----------------------------------------------------
        # FILTER BARIS
        # ----------------------------------------------------

        kata_abaikan = [
            "kode material",
            "kode",
            "total",
            "subtotal",
            "header",
            "item code",
            "part number",
            "k o d e"
        ]


        kode_lower = kode_material.lower().strip()


        if (
            not kode_material
            or kode_lower in kata_abaikan
        ):
            continue


        # ----------------------------------------------------
        # SIMPAN
        # ----------------------------------------------------

        rekap_data.append([
            target_name,
            kode_material,
            kode_stock,
            nama_material,
            spesifikasi,
            valid_qty,
            uom_car,
            valid_allowance,
            valid_pr_total
        ])

        baris_terproses += 1


    print(
        f"  -> {baris_terproses} baris terambil."
    )


# ============================================================
# 9. DATAFRAME
# ============================================================

df_rekap = pd.DataFrame(
    rekap_data,
    columns=[
        "Nama Sheet Sumber",
        "Kode Material",
        "Kode Material Stock",
        "Material / Komponen",
        "Spesifikasi",
        "Qty / TS (Series)",
        "UoM/Car",
        "Qty Total (Series) + Allowance",
        "QTY PR TOTAL"
    ]
)


# ============================================================
# 10. RINGKASAN
# ============================================================

print("\n--- RINGKASAN HASIL PER SHEET ---")

if not df_rekap.empty:

    print(
        df_rekap
        .groupby("Nama Sheet Sumber")
        .size()
    )

else:

    print("Tidak ada data yang berhasil diambil.")


# ============================================================
# 11. TULIS KE GOOGLE SHEETS
# ============================================================

nama_tab_rekap = (
    "REKAP_BOM_TOTAL (Berdasarkan Kode Material) 2"
)


try:

    sheet_rekap = spreadsheet.worksheet(
        nama_tab_rekap
    )

    sheet_rekap.clear()


except gspread.exceptions.WorksheetNotFound:

    sheet_rekap = spreadsheet.add_worksheet(
        title=nama_tab_rekap,
        rows="8000",
        cols="9"
    )


data_ke_sheets = (
    [df_rekap.columns.values.tolist()]
    + df_rekap.values.tolist()
)


sheet_rekap.update(
    data_ke_sheets
)


print(
    f"\n[SUKSES] TOTAL "
    f"{len(df_rekap)} baris data berhasil "
    f"ditarik dan disimpan ke sheet "
    f"'{nama_tab_rekap}'."
)

[INFO] Memetakan 29 dari 29 target sheet.


Sheet 'Elektrik + Pre Series'
  Header ditemukan pada baris 6
  Jumlah header terdeteksi: 8
  Mapping: {'kode': 4, 'kode_stock': 5, 'nama': 12, 'spek': 13, 'qty': 30, 'uom': 35, 'qty_allowance': 34, 'qty_pr_total': 44}
  -> 354 baris terambil.

Sheet 'De-Scoping'
  Header ditemukan pada baris 4
  Jumlah header terdeteksi: 6
  Mapping: {'kode': 4, 'kode_stock': 6, 'nama': None, 'spek': None, 'qty': 30, 'uom': 36, 'qty_allowance': 34, 'qty_pr_total': 62}
  -> 745 baris terambil.

Sheet 'Interior'
  Header ditemukan pada baris 6
  Jumlah header terdeteksi: 8
  Mapping: {'kode': 4, 'kode_stock': 5, 'nama': 12, 'spek': 13, 'qty': 30, 'uom': 35, 'qty_allowance': 34, 'qty_pr_total': 44}
  -> 339 baris terambil.

Sheet 'TMS'
  Header ditemukan pada baris 6
  Jumlah header terdeteksi: 8
  Mapping: {'kode': 4, 'kode_stock': 5, 'nama': 12, 'spek': 13, 'qty': 30, 'uom': 35, 'qty_allowance': 34, 'qty_pr_total': 44}
  -> 21 baris terambil.

Sheet 'CKD VVVF

# Read data

In [9]:
urx = "https://docs.google.com/spreadsheets/d/1Ibki18gicAFziEx1urTrvDv_KkzrkMMnRrbwqNYpOkw/export?format=csv&gid=1134789511"

data = pd.read_csv(urx, low_memory=False)
data.head()

,Nama Sheet Sumber,Kode Material,Kode Material Stock,Material / Komponen,Spesifikasi,Qty / TS (Series),UoM/Car,Qty Total (Series) + Allowance,QTY PR TOTAL
0,Elektrik + Pre Series,B336E12101,NaN,ELVP TC,Refer to dwg. 33.6-E12101 & 86.2-E12104 86.2-E...,2.0,set,32.0,32.0
1,Elektrik + Pre Series,B336E12201,NaN,ELVP M1,Refer to dwg. 33.6-E12201 & 86.2-E12204 86.2-E...,3.0,set,48.0,48.0
2,Elektrik + Pre Series,B336E12301,NaN,ELVP M2,Refer to dwg. 33.6-E12301 & 86.2-E12304 86.2-E...,3.0,set,48.0,48.0
3,Elektrik + Pre Series,B336E12401,NaN,ELVP T1,"Refer to dwg. 33.6-E12401 & 86.2-E12404, 86.2...",2.0,set,32.0,32.0
4,Elektrik + Pre Series,B336E12501,NaN,ELVP T2,Refer to dwg. 33.6-E12501 & 86.2-E12504 86.2-E...,1.0,set,16.0,16.0


# Analyze data

In [10]:
data[data["Nama Sheet Sumber"] == "Consumable Series"]

,Nama Sheet Sumber,Kode Material,Kode Material Stock,Material / Komponen,Spesifikasi,Qty / TS (Series),UoM/Car,Qty Total (Series) + Allowance,QTY PR TOTAL
5494,Consumable Series,D51QC0048,NaN,NaN,DCOTA 48ROLL/PA,120.0,pcs,1920.0,3840.0
5495,Consumable Series,D68QH10081,NaN,NaN,L-243 50ML,48.0,pcs,384.0,384.0
5496,Consumable Series,D68QH577050,NaN,NaN,L-577 50ML,72.0,pcs,1152.0,1152.0
5497,Consumable Series,D68QH211-206#WT,NaN,NaN,@20 X 600 ML 20 UP600,72.0,Sausage,576.0,580.0
5498,Consumable Series,D68QH255-206#BCF,NaN,NaN,@20 X 600 ML (BONDING),126.0,Sausage,1008.0,1008.0
...,...,...,...,...,...,...,...,...,...
5667,Consumable Series,D63SA0320,NaN,NaN,NaN,20.0,lembar,0.0,0.0
5668,Consumable Series,D63SA1200,NaN,NaN,NaN,20.0,lembar,0.0,0.0
5669,Consumable Series,D64QF00004-DNP,NaN,NaN,NaN,3.0,liter,0.0,0.0
5670,Consumable Series,D64QF00002,NaN,NaN,NaN,3.0,liter,0.0,0.0


In [11]:
ckdpantographData = data[data["Nama Sheet Sumber"] == "CKD Pantograph"]
# ckdpantographData[ckdpantographData["Material / Komponen"].isna()]
ckdpantographData.head()

,Nama Sheet Sumber,Kode Material,Kode Material Stock,Material / Komponen,Spesifikasi,Qty / TS (Series),UoM/Car,Qty Total (Series) + Allowance,QTY PR TOTAL
1606,CKD Pantograph,B52TO1213,NaN,Pantograph System (Without CKD Local Parts),Sesuai dengan spesifikasi teknis Pantograph no...,1.0,TS,6.0,6.0
1607,CKD Pantograph,B52TO2351,NaN,Joint,Spesifikasi sesuai dengan drawing no. P2132312-1,24.0,Pcs,144.0,144.0
1608,CKD Pantograph,B37PC000001,NaN,Bushing,Spesifikasi sesuai dengan drawing no. B286895-1,12.0,Pcs,72.0,72.0
1609,CKD Pantograph,B45MG0001,NaN,Pin,Spesifikasi sesuai dengan drawing no. P2132304-1,24.0,Pcs,144.0,144.0
1610,CKD Pantograph,B44KO0009,NaN,Washer,Spesifikasi sesuai dengan drawing no. P2173012-8,24.0,Pcs,144.0,144.0


In [12]:
ckdsivdata = data[data["Nama Sheet Sumber"] == "CKD SIV"]
ckdsivdata[ckdsivdata["Material / Komponen"].isna()]

,Nama Sheet Sumber,Kode Material,Kode Material Stock,Material / Komponen,Spesifikasi,Qty / TS (Series),UoM/Car,Qty Total (Series) + Allowance,QTY PR TOTAL
1977,CKD SIV,B42CD0416,NaN,NaN,NaN,0.0,NaN,0.0,0.0
1982,CKD SIV,B44LA0008,NaN,NaN,NaN,0.0,NaN,100.0,100.0


In [13]:
carbodydata = data[data["Nama Sheet Sumber"] == "Carbody"]
carbodydata[carbodydata["Material / Komponen"].isna()].head()

,Nama Sheet Sumber,Kode Material,Kode Material Stock,Material / Komponen,Spesifikasi,Qty / TS (Series),UoM/Car,Qty Total (Series) + Allowance,QTY PR TOTAL


In [14]:
fasteningmekanikdata = data[data["Nama Sheet Sumber"] == "Fastening Mekanik"]
fasteningmekanikdata[fasteningmekanikdata["Material / Komponen"].isna()].head()

,Nama Sheet Sumber,Kode Material,Kode Material Stock,Material / Komponen,Spesifikasi,Qty / TS (Series),UoM/Car,Qty Total (Series) + Allowance,QTY PR TOTAL


In [15]:
rrmdata = data[data["Nama Sheet Sumber"] == "Realisasi Raw Material"]
rrmdata[rrmdata["Material / Komponen"].isna()]

,Nama Sheet Sumber,Kode Material,Kode Material Stock,Material / Komponen,Spesifikasi,Qty / TS (Series),UoM/Car,Qty Total (Series) + Allowance,QTY PR TOTAL
3254,Realisasi Raw Material,A10ND0003,NaN,NaN,NaN,0.0,SHT,0.0,0.0
3255,Realisasi Raw Material,A11AB0002,NaN,NaN,NaN,0.0,SHT,0.0,0.0
3256,Realisasi Raw Material,A11AB0012,NaN,NaN,NaN,0.0,SHT,0.0,0.0
3257,Realisasi Raw Material,A11AB0016,NaN,NaN,NaN,0.0,SHT,0.0,0.0
3258,Realisasi Raw Material,A11AB0023,NaN,NaN,NaN,0.0,SHT,0.0,0.0
...,...,...,...,...,...,...,...,...,...
3372,Realisasi Raw Material,A11FD0032,NaN,NaN,NaN,0.0,SHT,0.0,0.0
3373,Realisasi Raw Material,A11XY0040D9,NaN,NaN,NaN,0.0,SHT,0.0,0.0
3374,Realisasi Raw Material,A11AA0045,NaN,NaN,NaN,0.0,SHT,0.0,0.0
3375,Realisasi Raw Material,A11AA0060,NaN,NaN,NaN,0.0,SHT,0.0,0.0


In [16]:
result = data[data["Material / Komponen"].isna()]
result.groupby("Nama Sheet Sumber").size()

,0
Nama Sheet Sumber,
CKD SIV,2
Consumable Series,178
De-Scoping,745
Realisasi Raw Material,123
Sistem Mekanik,497


In [17]:
spesifikasiisnaresult = data[data["Spesifikasi"].isna()]
spesifikasiisnaresult.groupby("Nama Sheet Sumber").size()

,0
Nama Sheet Sumber,
Bogie KIT,1
CKD SIV,2
Consumable Pre Series,2
Consumable Series,14
Consumable Tools,11
Crashwrothiness,1
De-Scoping,745
Elektrik,55
Elektrik + Pre Series,57


In [18]:
bkdata = data[data["Nama Sheet Sumber"] == "Bogie KIT"]
bkdata[bkdata["Spesifikasi"].isna()]

,Nama Sheet Sumber,Kode Material,Kode Material Stock,Material / Komponen,Spesifikasi,Qty / TS (Series),UoM/Car,Qty Total (Series) + Allowance,QTY PR TOTAL
3138,Bogie KIT,B52TP3101,NaN,Eathing Brush,NaN,24.0,NaN,384.0,384.0


In [53]:
ckdpdata = data[data["Nama Sheet Sumber"] == "CKD Pantograph"]
ckdpdata[ckdpdata["Spesifikasi"].isna()]

,Nama Sheet Sumber,Kode Material,Kode Material Stock,Material / Komponen,Spesifikasi,Qty / TS (Series),UoM/Car,Qty Total (Series) + Allowance,QTY PR TOTAL


In [54]:
cdata = data[data["Nama Sheet Sumber"] == "Carbody"]
cdata[cdata["Spesifikasi"].isna()]

,Nama Sheet Sumber,Kode Material,Kode Material Stock,Material / Komponen,Spesifikasi,Qty / TS (Series),UoM/Car,Qty Total (Series) + Allowance,QTY PR TOTAL


In [55]:
cpsdata = data[data["Nama Sheet Sumber"] == "Consumable Pre Series"]
cpsdata[cpsdata["Spesifikasi"].isna()]
#bermasalah nih kalkulasi quantitynya.

,Nama Sheet Sumber,Kode Material,Kode Material Stock,Material / Komponen,Spesifikasi,Qty / TS (Series),UoM/Car,Qty Total (Series) + Allowance,QTY PR TOTAL
6987,Consumable Pre Series,D66UQ0001#BC,NaN,LAKBAN HITAM,NaN,0.0,Roll,0.0,0.0
7058,Consumable Pre Series,D68QH0125,NaN,ANTISOL E 125,NaN,0.0,liter,0.0,0.0


In [20]:
csdata = data[data["Nama Sheet Sumber"] == "Consumable Series"]
csdata[csdata["Spesifikasi"].isna()] #bermasalah nih ada beberapa data yang nggak masuk

,Nama Sheet Sumber,Kode Material,Kode Material Stock,Material / Komponen,Spesifikasi,Qty / TS (Series),UoM/Car,Qty Total (Series) + Allowance,QTY PR TOTAL
5503,Consumable Series,D66UQ0001#BC,NaN,NaN,NaN,408.0,Roll,0.0,0.0
5573,Consumable Series,D68QH0125,NaN,NaN,NaN,60.0,liter,0.0,0.0
5597,Consumable Series,D82WL0001,NaN,NaN,NaN,120.0,Roll,1080.0,1080.0
5617,Consumable Series,D68QH1004,NaN,NaN,NaN,48.0,TB,384.0,384.0
5628,Consumable Series,D66UQ0001#BC,NaN,NaN,NaN,12.0,Roll,0.0,0.0
5635,Consumable Series,D31WH0506,NaN,NaN,NaN,12.0,Liter,96.0,96.0
5650,Consumable Series,D67UK0002,NaN,NaN,NaN,24.0,Kg,48.0,48.0
5664,Consumable Series,D68QG0005,NaN,NaN,NaN,6.0,kg,0.0,0.0
5665,Consumable Series,D62QE0088#HB,NaN,NaN,NaN,10.0,literset,0.0,0.0
5666,Consumable Series,D63SA0017,NaN,NaN,NaN,20.0,lembar,0.0,0.0


In [ ]:
csdata = data[(data["Nama Sheet Sumber"] == "Consumable Series") & (data["Kode Material"] == "D66UG0048-NCI")]
csdata

In [ ]:
crsdata = data[data["Nama Sheet Sumber"] == "Crashwrothiness"]
crsdata[crsdata["Spesifikasi"].isna()]

In [ ]:
dsdata = data[data["Nama Sheet Sumber"] == "De-Scoping"]
dsdata[dsdata["Spesifikasi"].isna()]

In [ ]:

elektrikData = data[data["Nama Sheet Sumber"] == "Elektrik"]
elektrikData[elektrikData["Spesifikasi"].isna()]

In [ ]:
elektrikData = data[data["Nama Sheet Sumber"] == "Elektrik + Pre Series"]
elektrikData[elektrikData["Spesifikasi"].isna()]

In [ ]:
fmdata = data[data["Nama Sheet Sumber"] == "Fastening Mekanik"]
fmdata[fmdata["Spesifikasi"].isna()]

In [ ]:
jtdata = data[data["Nama Sheet Sumber"] == "Jig Tool"]
jtdata[jtdata["Spesifikasi"].isna()]

In [ ]:
jt6data = data[data["Nama Sheet Sumber"] == "Jig Tools 6TS Pelokalan"]
jt6data[jt6data["Spesifikasi"].isna()] #Bermasalah ini di Qty / TS

In [ ]:
jt6data = data[data["Nama Sheet Sumber"] == "Jig Tools 6TS Pelokalan"]
jt6data[jt6data["Qty / TS"] != 0.0]

In [ ]:
kudata = data[data["Nama Sheet Sumber"] == "Komponen Utama"]
kudata[kudata["Spesifikasi"].isna()]

In [ ]:
rawdata = data[data["Nama Sheet Sumber"] == "Realisasi Raw Material"]
rawdata[rawdata["Spesifikasi"].isna()]

In [ ]:
smdata = data[data["Nama Sheet Sumber"] == "Sistem Mekanik"]
smdata[smdata["Spesifikasi"].isna()] #Anomali nih Qty nya

In [ ]:
smdata = data[data["Nama Sheet Sumber"] == "Sistem Mekanik"]
smdata[smdata["Qty / TS"] != 0.0]

In [ ]:
toolsData = data[data["Nama Sheet Sumber"] == "Tools"]
toolsData[toolsData["Spesifikasi"].isna()] #Qty nya juga salah ini, nggak akurat

In [ ]:
toolsData = data[data["Nama Sheet Sumber"] == "Tools"]
toolsData[toolsData["Qty / TS"] != 0.0]

In [ ]:
tctpdata = data[data["Nama Sheet Sumber"] == "Tools & Consumable Tools Perbaikan"]
tctpdata[tctpdata["Spesifikasi"].isna()]

In [ ]:
tctpdata = data[data["Nama Sheet Sumber"] == "Tools & Consumable Tools Perbaikan"]
tctpdata[tctpdata["Qty / TS"] != 0.0]

In [ ]:
zeroquantitydata = data[data["Qty / TS"] == 0.0]
zeroquantitydata.groupby("Nama Sheet Sumber").size()

- Consumable Pre Series (Masalah di quantity)
- Consumable Series (Ada data yang nggak masuk)
- Jig Tools 6TS Pelokalan (Bermasalah di QTY / TS)
- Sistem Mekanik (Bermasalah di QTY / TS)
- Tools (bermasalah di qty / ts)


# New Section

In [ ]:
bomdata = data.groupby("Nama Sheet Sumber").size()
bomdata = bomdata.to_frame(name="Jumlah")
bomdata

,Jumlah
Nama Sheet Sumber,
Bogie KIT,44
CKD Pantograph,266
CKD SIV,119
CKD VVVF,147
Carbody,246
Consumable Pre Series,93
Consumable Series,178
Consumable Tools,275
Crashwrothiness,19


In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [ ]:
data[data['Material / Komponen'].isna() & (data['Qty / TS'] == 0)]

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
1235,CKD SIV,B42CD0416,NaN,NaN,0.0
2677,Carbody,BRACKET,NaN,NaN,0.0


In [ ]:
bogieKitData = data[(data['Nama Sheet Sumber'] == 'Bogie KIT')]
bogieKitData

,Nama Sheet Sumber,Kode Material,Material / Komponen,Spesifikasi,Qty / TS
2374,Bogie KIT,B48BR0100,Driven Wheelset (MB),Sesuai dengan spesifikasi teknis 108/SPT/H100...,0.0
2375,Bogie KIT,B48BL0106,Gear Unit / Gear Box,Sesuai dengan spesifikasi teknis 106/SPT/H100...,0.0
2376,Bogie KIT,B48BC0119,Axle Box Housing,"02.0-M20002, 119/SPT/H1005MB051/22",0.0
2377,Bogie KIT,B48LA5959,Journal roller bearing,Sesuai dengan spesifikasi teknis 130/SPT/H1005...,0.0
2378,Bogie KIT,B48AH2004,Rubber Bush for Axle Box,Sesuai dengan spesifikasi teknis 119/SPT/H1005...,0.0
2379,Bogie KIT,B48BW1622,Axle Box Cap (for Grounding Device),02.0-M20006,0.0
2380,Bogie KIT,B48RD0015,Pole Wheel,02.0-E11014,0.0
2381,Bogie KIT,B48BW1122,Cap Cover,02.0-P23311,0.0
2382,Bogie KIT,B48BQ1122,Coil Spring,Sesuai dengan spesifikasi teknis 129/SPT/H10...,0.0
2383,Bogie KIT,B48DJ1122,Axle Spring Guide (2),Sesuai drawing B48DJ1122,0.0


# Material Dataset from SAP

In [21]:
sapdataseturl = "https://docs.google.com/spreadsheets/d/1jnuEazMkGxbXcvP2mvvGlRZQZR-N0FMPf3aNhw5lH7Q/export?format=csv&gid=826586568"

sapdataset = pd.read_csv(sapdataseturl, low_memory=False)
sapdataset.columns = sapdataset.columns.str.strip()

In [22]:
sapdataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108300 entries, 0 to 108299
Data columns (total 27 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   WBS Elem          108296 non-null  object 
 1   WBS Description   108296 non-null  object 
 2   Purch.Req.        108296 non-null  float64
 3   Del. Indicator    1 non-null       object 
 4   PR Status         108296 non-null  object 
 5   Req.Date          108296 non-null  object 
 6   PR Deliv. Dat     108296 non-null  object 
 7   Chngd on          108296 non-null  object 
 8   Kode Material     102715 non-null  object 
 9   Mat. Description  108296 non-null  object 
 10  Spesifikasi       101889 non-null  object 
 11  Qty Requested     108296 non-null  object 
 12  Un                108296 non-null  object 
 13  Ordered           108296 non-null  object 
 14  PO Number         55421 non-null   object 
 15  PO Date           108296 non-null  object 
 16  PO Deliv. Dat     10

In [23]:
sapdataset = sapdataset[[
    "WBS Description",
    "Kode Material",
    "Mat. Description",
    "Spesifikasi",
    "Purch.Req.",
    "PR Status",
    "Req.Date",
    "PR Deliv. Dat",
    "Chngd on",
    "Qty Requested",
    "Un",
    "PO Number",
    "PO Date",
    "PO Deliv. Dat",
    "Ordered",
    "Created by",
    "Requested By",
    "Matl Group"
  ]]

In [24]:
sapdataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108300 entries, 0 to 108299
Data columns (total 18 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   WBS Description   108296 non-null  object 
 1   Kode Material     102715 non-null  object 
 2   Mat. Description  108296 non-null  object 
 3   Spesifikasi       101889 non-null  object 
 4   Purch.Req.        108296 non-null  float64
 5   PR Status         108296 non-null  object 
 6   Req.Date          108296 non-null  object 
 7   PR Deliv. Dat     108296 non-null  object 
 8   Chngd on          108296 non-null  object 
 9   Qty Requested     108296 non-null  object 
 10  Un                108296 non-null  object 
 11  PO Number         55421 non-null   object 
 12  PO Date           108296 non-null  object 
 13  PO Deliv. Dat     108296 non-null  object 
 14  Ordered           108296 non-null  object 
 15  Created by        108296 non-null  object 
 16  Requested By      10

In [25]:
#grouping data based on kode Material
komatUniqueData = sapdataset.groupby("Kode Material").size().to_frame(name="Jumlah").head()
komatUniqueData[komatUniqueData["Jumlah"] == 2]

,Jumlah
Kode Material,
561U03131,2


In [26]:
# Contoh data Material yang sama namun no pr nya berbeda.
sapdataset[sapdataset["Kode Material"] == "B52TF0009"]

,WBS Description,Kode Material,Mat. Description,Spesifikasi,Purch.Req.,PR Status,Req.Date,PR Deliv. Dat,Chngd on,Qty Requested,Un,PO Number,PO Date,PO Deliv. Dat,Ordered,Created by,Requested By,Matl Group
24996,PENGADAAN 16 TS KRL KCI (16 TC1),B52TF0009,Line Flow Fan Kabin,"Mitsubishi Electric, LD-08WB, Single axial t...",10024908.0,B,14.06.2024,21.08.2024,16.06.2024,4,PC,4400000826,05.11.2024,04.02.2025,4,PPC_RENP07,Memo,PRTELIN
24997,PENGADAAN 16 TS KRL KCI (16 TC1),B52TF0009,Line Flow Fan Kabin,"Mitsubishi Electric, LD-08WB, Single axial t...",10024908.0,B,14.06.2024,31.10.2024,16.06.2024,4,PC,4400000826,05.11.2024,04.02.2025,4,PPC_RENP07,Memo,PRTELIN
24998,PENGADAAN 16 TS KRL KCI (16 TC1),B52TF0009,Line Flow Fan Kabin,"Mitsubishi Electric, LD-08WB, Single axial t...",10024908.0,B,14.06.2024,19.01.2025,16.06.2024,4,PC,4400000826,05.11.2024,04.02.2025,4,PPC_RENP07,Memo,PRTELIN
24999,PENGADAAN 16 TS KRL KCI (16 TC1),B52TF0009,Line Flow Fan Kabin,"Mitsubishi Electric, LD-08WB, Single axial t...",10024908.0,B,14.06.2024,07.04.2025,16.06.2024,4,PC,4400000826,05.11.2024,04.02.2025,4,PPC_RENP07,Memo,PRTELIN
25004,PENGADAAN 16 TS KRL KCI (16 TC2),B52TF0009,Line Flow Fan Kabin,"Mitsubishi Electric, LD-08WB, Single axial t...",10024909.0,B,14.06.2024,21.08.2024,16.06.2024,4,PC,4400000826,05.11.2024,30.11.2024,4,PPC_RENP07,Memo,PRTELIN
25005,PENGADAAN 16 TS KRL KCI (16 TC2),B52TF0009,Line Flow Fan Kabin,"Mitsubishi Electric, LD-08WB, Single axial t...",10024909.0,B,14.06.2024,31.10.2024,16.06.2024,4,PC,4400000826,05.11.2024,04.02.2025,4,PPC_RENP07,Memo,PRTELIN
25006,PENGADAAN 16 TS KRL KCI (16 TC2),B52TF0009,Line Flow Fan Kabin,"Mitsubishi Electric, LD-08WB, Single axial t...",10024909.0,B,14.06.2024,19.01.2025,16.06.2024,4,PC,4400000826,05.11.2024,04.02.2025,4,PPC_RENP07,Memo,PRTELIN
25007,PENGADAAN 16 TS KRL KCI (16 TC2),B52TF0009,Line Flow Fan Kabin,"Mitsubishi Electric, LD-08WB, Single axial t...",10024909.0,B,14.06.2024,07.04.2025,16.06.2024,4,PC,4400000826,05.11.2024,04.02.2025,4,PPC_RENP07,Memo,PRTELIN
44711,PENGADAAN 16 TS KRL KCI,B52TF0009,Line Flow Fan Kabin,"Mitsubishi Electric, LD-08WB, Single axial t...",10029619.0,B,13.06.2025,13.08.2025,16.06.2025,3,PC,4400000996,01.08.2025,30.11.2025,3,PPC_RENP07,BA-Rusak,PRTELIN
48851,PENGADAAN 16 TS KRL KCI,B52TF0009,Line Flow Fan Kabin,"Mitsubishi Electric, LD-08WB, Single axial t...",10030498.0,B,07.08.2025,30.09.2025,29.08.2025,1,PC,4400001017,11.09.2025,30.01.2026,1,PPC_RENP07,Memo-Spare,PRTELIN


In [27]:
# Contoh data Material yang berbeda namun no pr nya sama.
noprUniqueData = sapdataset.groupby("Purch.Req.").size().to_frame(name="Jumlah").head()
# noprUniqueData[noprUniqueData["Jumlah"] == 5]
noprUniqueData.head()

,Jumlah
Purch.Req.,
10009676.0,2
10009678.0,6
10009705.0,3
10009706.0,5
10009724.0,9


In [28]:
sapdataset[sapdataset["Purch.Req."] == 10009706.0]

,WBS Description,Kode Material,Mat. Description,Spesifikasi,Purch.Req.,PR Status,Req.Date,PR Deliv. Dat,Chngd on,Qty Requested,Un,PO Number,PO Date,PO Deliv. Dat,Ordered,Created by,Requested By,Matl Group
15,PENGADAAN SARANA 186 CAR LRT JABODEBEK,A11RN0100E10,PLATE,"AL 6061-T6, T. 10 X 1500mm x3000mm",10009706.0,B,23.02.2018,26.09.2018,24.04.2018,50,SHT,4500003722,25.04.2018,16.07.2018,50,INKA_REN_PR2,ANdA,RMRAWM
102,PENGADAAN SARANA 186 CAR LRT JABODEBEK,A21RP6578,ANGLE L,"AI 6005A T-6, L=6000mm, drawing 21.0-U0300101...",10009706.0,B,23.02.2018,26.09.2018,24.04.2018,118,STK,4500003847,15.05.2018,07.08.2018,118,INKA_REN_PR2,ANdA,RMRAWM
103,PENGADAAN SARANA 186 CAR LRT JABODEBEK,A17RP8008005,SQUARE HOLLOW,"AI 6005A T-6, L=6000mm, drawing 21.0-U0300101...",10009706.0,B,23.02.2018,26.09.2018,24.04.2018,137,STK,4500003847,15.05.2018,07.08.2018,137,INKA_REN_PR2,ANdA,RMRAWM
104,PENGADAAN SARANA 186 CAR LRT JABODEBEK,A17RP80040028,RECTANGULAR HOLLOW,"AI 6005A T-6, L=6000mm, drawing 21.0-U0300101...",10009706.0,B,23.02.2018,26.09.2018,24.04.2018,100,STK,4500003847,15.05.2018,07.08.2018,100,INKA_REN_PR2,ANdA,RMRAWM
105,PENGADAAN SARANA 186 CAR LRT JABODEBEK,A23RP45505,CHANNEL,"AI 6005A T-6, L=6000mm, drawing 21.0-U0300101...",10009706.0,B,23.02.2018,26.09.2018,24.04.2018,50,STK,4500003847,15.05.2018,07.08.2018,50,INKA_REN_PR2,ANdA,RMRAWM


In [29]:
# Contoh data yang Kode Material & NO xPR sama
noprKomatUniqueData = sapdataset.groupby(["Kode Material", "Purch.Req."]).size().to_frame(name="Jumlah")
noprKomatUniqueData[noprKomatUniqueData['Jumlah'] > 1].head()

Jumlah
Kode Material Purch.Req.        
A01LK0020     10027330.0       2
              10032575.0       2
              10032960.0       2
              10033983.0       5
A01UL0040     10025206.0       7

In [30]:
len(noprKomatUniqueData)

75395

In [31]:
sapdataset[(sapdataset["Purch.Req."] == 10024908) & (sapdataset["Kode Material"] == "B52TF0009")]

,WBS Description,Kode Material,Mat. Description,Spesifikasi,Purch.Req.,PR Status,Req.Date,PR Deliv. Dat,Chngd on,Qty Requested,Un,PO Number,PO Date,PO Deliv. Dat,Ordered,Created by,Requested By,Matl Group
24996,PENGADAAN 16 TS KRL KCI (16 TC1),B52TF0009,Line Flow Fan Kabin,"Mitsubishi Electric, LD-08WB, Single axial t...",10024908.0,B,14.06.2024,21.08.2024,16.06.2024,4,PC,4400000826,05.11.2024,04.02.2025,4,PPC_RENP07,Memo,PRTELIN
24997,PENGADAAN 16 TS KRL KCI (16 TC1),B52TF0009,Line Flow Fan Kabin,"Mitsubishi Electric, LD-08WB, Single axial t...",10024908.0,B,14.06.2024,31.10.2024,16.06.2024,4,PC,4400000826,05.11.2024,04.02.2025,4,PPC_RENP07,Memo,PRTELIN
24998,PENGADAAN 16 TS KRL KCI (16 TC1),B52TF0009,Line Flow Fan Kabin,"Mitsubishi Electric, LD-08WB, Single axial t...",10024908.0,B,14.06.2024,19.01.2025,16.06.2024,4,PC,4400000826,05.11.2024,04.02.2025,4,PPC_RENP07,Memo,PRTELIN
24999,PENGADAAN 16 TS KRL KCI (16 TC1),B52TF0009,Line Flow Fan Kabin,"Mitsubishi Electric, LD-08WB, Single axial t...",10024908.0,B,14.06.2024,07.04.2025,16.06.2024,4,PC,4400000826,05.11.2024,04.02.2025,4,PPC_RENP07,Memo,PRTELIN


# Simulation

In [215]:
komat = "B55TJ0012"

#Data yang berada di dataset SAP.
sapdataset[sapdataset["Material"] == komat]

,WBS Description,Material,Mat. Description,Spesifikasi,Purch.Req.,PR Status,Req.Date,PR Deliv. Dat,Chngd on,Qty Requested,Un,PO Number,PO Date,PO Deliv. Dat,Ordered,Created by,Requested By,Matl Group
24991,PENGADAAN 16 TS KRL KCI (16 TC1),B55TJ0012,Smoke and Heat Detector,"Smoke and heat detector, tegangan supply 110 ...",10024908.0,B,14.06.2024,10.07.2024,16.06.2024,3.0,PC,4400000846,08.12.2024,19.02.2025,3,PPC_RENP07,Memo,PRTELIN
24992,PENGADAAN 16 TS KRL KCI (16 TC1),B55TJ0012,Smoke and Heat Detector,"Smoke and heat detector, tegangan supply 110 ...",10024908.0,B,14.06.2024,21.08.2024,16.06.2024,9.0,PC,4400000846,08.12.2024,19.02.2025,9,PPC_RENP07,Memo,PRTELIN
24993,PENGADAAN 16 TS KRL KCI (16 TC1),B55TJ0012,Smoke and Heat Detector,"Smoke and heat detector, tegangan supply 110 ...",10024908.0,B,14.06.2024,31.10.2024,16.06.2024,12.0,PC,4400000846,08.12.2024,19.02.2025,12,PPC_RENP07,Memo,PRTELIN
24994,PENGADAAN 16 TS KRL KCI (16 TC1),B55TJ0012,Smoke and Heat Detector,"Smoke and heat detector, tegangan supply 110 ...",10024908.0,B,14.06.2024,19.01.2025,16.06.2024,12.0,PC,4400000846,08.12.2024,19.02.2025,12,PPC_RENP07,Memo,PRTELIN
24995,PENGADAAN 16 TS KRL KCI (16 TC1),B55TJ0012,Smoke and Heat Detector,"Smoke and heat detector, tegangan supply 110 ...",10024908.0,B,14.06.2024,07.04.2025,16.06.2024,12.0,PC,4400000846,08.12.2024,19.02.2025,12,PPC_RENP07,Memo,PRTELIN
25000,PENGADAAN 16 TS KRL KCI (16 TC2),B55TJ0012,Smoke and Heat Detector,"Smoke and heat detector, tegangan supply 110 ...",10024909.0,B,14.06.2024,21.08.2024,16.06.2024,12.0,PC,4400000846,08.12.2024,19.02.2025,12,PPC_RENP07,Memo,PRTELIN
25001,PENGADAAN 16 TS KRL KCI (16 TC2),B55TJ0012,Smoke and Heat Detector,"Smoke and heat detector, tegangan supply 110 ...",10024909.0,B,14.06.2024,31.10.2024,16.06.2024,12.0,PC,4400000846,08.12.2024,19.02.2025,12,PPC_RENP07,Memo,PRTELIN
25002,PENGADAAN 16 TS KRL KCI (16 TC2),B55TJ0012,Smoke and Heat Detector,"Smoke and heat detector, tegangan supply 110 ...",10024909.0,B,14.06.2024,19.01.2025,16.06.2024,12.0,PC,4400000846,08.12.2024,19.02.2025,12,PPC_RENP07,Memo,PRTELIN
25003,PENGADAAN 16 TS KRL KCI (16 TC2),B55TJ0012,Smoke and Heat Detector,"Smoke and heat detector, tegangan supply 110 ...",10024909.0,B,14.06.2024,07.04.2025,16.06.2024,12.0,PC,4400000846,08.12.2024,19.02.2025,12,PPC_RENP07,Memo,PRTELIN
25008,PENGADAAN 16 TS KRL KCI (48 M1),B55TJ0012,Smoke and Heat Detector,"Smoke and heat detector, tegangan supply 110 ...",10024910.0,B,14.06.2024,10.07.2024,16.06.2024,2.0,PC,4400000830,08.11.2024,31.12.2024,2,PPC_RENP07,Memo,PRTELIN


In [216]:
x = sapdataset[sapdataset["Material"] == komat]
len(x) #jumlah total PR berapa kali.

34

In [217]:
#data yang berada di dataset SAP untuk kode komat tertentu

sapdataset['Qty Requested'] = pd.to_numeric(sapdataset['Qty Requested'], errors='coerce').fillna(0)
total_qty = sapdataset.loc[sapdataset['Material'] == komat, 'Qty Requested'].sum()

print(f"Total Quantity PR untuk {komat} adalah:", total_qty)

Total Quantity PR untuk B55TJ0012 adalah: 416.0


In [222]:
#Data yang berada di dataset BOM.
total_sap = total_qty
total_bom = data[data["Kode Material"] == komat]["QTY PR TOTAL"]
print(f"{total_sap} | {total_bom}")

416.0 | 2502    416.0
Name: QTY PR TOTAL, dtype: float64


* Berarti sebuah data material dikatakan sudah PR, Ketika data Qty PR Total pada data material yang ada di daftar Kebutuhan Material (BOM) sama dengan kalkulasi Total quantity PR berdasarkan data yang ada di dataset SAP.

**Data data yang diperlukan untuk Keperluan Dashboard Analytics**

Contoh data yang digunakan adalah data dengan kode material "B55TJ0012" di BOM Komponen Utama.

Data-data yang digunakan untuk dataset Daftar Kebutuhan Material / BOM

*   Kode Material
*   Kode Material Stock
*   Material / Komponen
*   Spesifikasi
*   Qty / TS (Series)
*   QTY PR
*   No PR
*   Qty Total + Allowance

Data-data yang digunakan untuk dataset SAP

* WBS Description,
* Material (Kode Material),
* Mat. Description,
* Spesifikasi,
* Purch.Req.,
* PR Status,
* Req.Date,
* PR Deliv. Dat,
* Chngd on,
* Qty Requested,
* Un,
* PO Number,
* PO Date,
* PO Deliv. Dat,
* Ordered,
* Created by,
* Requested By,
* Matl Group

Data-data yang digunakan untuk match antara Daftar Kebutuhan Material / BOM dan SAP

* Kode Material,
* PR Number
* PO Number

# Purchase Requisiton PIE CHART

In [32]:
komat = "A01LK0020"

sapdataset['Qty Requested'] = pd.to_numeric(sapdataset['Qty Requested'], errors='coerce').fillna(0)
total_qty = sapdataset.loc[sapdataset['Kode Material'] == komat, 'Qty Requested'].sum()

print(f"Total Qty Requested untuk {komat} adalah:", total_qty)

Total Qty Requested untuk A01LK0020 adalah: 2677.0


In [33]:
bomurl = "https://docs.google.com/spreadsheets/d/1Ibki18gicAFziEx1urTrvDv_KkzrkMMnRrbwqNYpOkw/export?format=csv&gid=1134789511"

bomdataset = pd.read_csv(bomurl, low_memory=False)

instockData = bomdataset[bomdataset["Kode Material Stock"].notna()]
nostockData = bomdataset[bomdataset["Kode Material Stock"].isna()]

In [34]:
instockData

,Nama Sheet Sumber,Kode Material,Kode Material Stock,Material / Komponen,Spesifikasi,Qty / TS (Series),UoM/Car,Qty Total (Series) + Allowance,QTY PR TOTAL
41,Elektrik + Pre Series,B52TP3212,B52TP3212@,Strain Relief,"Wago, 769-412 (5 poles)",12.0,Pcs,192.0,177.0
42,Elektrik + Pre Series,B52TI8231,B52TI8231@,Locking Levers,"Wago, 769-431",12.0,Pcs,192.0,0.0
59,Elektrik + Pre Series,B52TE6346,B52TE0860,Door Open Lamp/Head Signal Light Red,"APT, AD16-60KTKA-r, Red, 110 VDC",24.0,Pcs,384.0,380.0
60,Elektrik + Pre Series,B52TI0152,B52UM2521,SELECTOR SWITCH,"SCHNEIDER ELECTRIC, XB5-AD21, 1 No, 2 Pos",12.0,Pcs,192.0,192.0
62,Elektrik + Pre Series,A52ZG40950,A52ZG40950@,Main Circuit Cable,"95 mmSQ , 3600V, SpekTek: 307-SPT-K1003TB091-16",915.0,meter,14933.0,26093.0
...,...,...,...,...,...,...,...,...,...
6275,Jig Tool,B41BZ0516,B41BZ0516,HEXAGON SOCKET HEAD,M5 x 16 (Baut Kunci L),500.0,PCS,500.0,5.5
6280,Jig Tool,A11AB0100,A11AB0100,STEEL PLATE SS400,10 x 1200 x 2400,1.0,Sheet,1.0,0.0
6287,Jig Tool,A11AB0400,A11ST0400,STEEL PLATE SS400,40 x 1200 x 2400,1.0,Sheet,1.0,0.0
6288,Jig Tool,A15AB0200,A15AB0190,ROUNDBAR SS400,Dia20 x 6000,1.0,Batang,1.0,0.0


In [35]:
sapdataset['Qty Requested'] = pd.to_numeric(sapdataset['Qty Requested'], errors='coerce').fillna(0)
sapdataset['Ordered'] = pd.to_numeric(sapdataset['Ordered'], errors='coerce').fillna(0)

sapdataset_ringkas = sapdataset.groupby('Kode Material', as_index=False).agg({
    'Mat. Description': 'first', # Mengambil deskripsi pertama
    'Spesifikasi': 'first',      # Mengambil spesifikasi pertama
    'PR Status': 'first',
    'Qty Requested': 'sum',      # Menjumlahkan total Qty Requested
    'Ordered': 'sum'             # Menjumlahkan total Ordered
})

sapdataset_ringkas

,Kode Material,Mat. Description,Spesifikasi,PR Status,Qty Requested,Ordered
0,16.09.2026,PEMADAM KEBAKARAN OTOMATIS,Satu set sistem pemadam kebakaran otomatis te...,N,1.0,0.0
1,561U03131,Flexible hose from UC to MRP,DRAWING 56.1-U03131,B,124.0,124.0
2,A007A18507,Piping for cooling genset,"Sesuai BOM Raw Material Sistem Mekanik, 205-B...",N,2.0,0.0
3,A007A18508,Piping for Drainase,"Sesuai BOM Raw Material Sistem Mekanik, 205-B...",N,2.0,0.0
4,A01LK0020,Kawat Tali,GALVANIS Dia. 2mm,B,2677.0,2061.0
...,...,...,...,...,...,...
12261,E97SC2000,Printer Marking Tube dan Heatshrink,SUPVAN TP2000M (spek dan gambar terlampir),B,1.0,1.0
12262,E97SH0002,ISO Container 20 feet,ISO 668,B,1.0,1.0
12263,E97SH0003,ISO Container 40 feet,ISO 668,B,1.0,1.0
12264,E97TC1321,Rubber Hardness,Shore Durometer HH-336 Shore A Digital/Compact,B,1.0,1.0


In [36]:
df_gabung = pd.merge(nostockData, sapdataset_ringkas, on='Kode Material', how='inner')

x = df_gabung[df_gabung["QTY PR TOTAL"] != df_gabung["Qty Requested"]]

#data yang bisa dikatakan belum PR
x[x["QTY PR TOTAL"] > x["Qty Requested"]]

,Nama Sheet Sumber,Kode Material,Kode Material Stock,Material / Komponen,Spesifikasi_x,Qty / TS (Series),UoM/Car,Qty Total (Series) + Allowance,QTY PR TOTAL,Mat. Description,Spesifikasi_y,PR Status,Qty Requested,Ordered
57,Elektrik + Pre Series,A52ZG41500,NaN,Main Circuit Cable,"150 mmSQ , 3600V, SpekTek: 307-SPT-K1003TB091-16",1056.0,meter,17234.0,17500.0,Main Circuit Cable,"150 mmsq, 3600V, SpekTek: 307-SPT-K1003TB091-16",B,4800.0,4800.0
58,Elektrik + Pre Series,A52ZG40710,NaN,Main Circuit Cable,"70 mmSQ , 3600V, SpekTek: 307-SPT-K1003TB091-16",486.0,meter,7932.0,7932.0,Main Circuit Cable,"70 mmSQ , 3600V, SpekTek: 307-SPT-K1003TB091-16",B,5177.0,5177.0
60,Elektrik + Pre Series,A52YC1070,NaN,Main Circuit Cable,"Rev 0: LEONI/LS/HubSuhner/LAPP, 70 mmsq, shiel...",330.0,meter,5386.0,6000.0,Main Circuit Cable,"LEONI/LS/HubSuhner/LAPP, 70 mmsq, shielded, 1...",B,1200.0,1680.0
61,Elektrik + Pre Series,A52YC2262,NaN,Traction Motor Cables,"35 mmSQ Shielded, 1800V, SpekTek: 307-SPT-K100...",282.0,meter,4603.0,4600.0,Traction Motor Cables,"35 mmSQ Shielded, 1800V, SpekTek: 307-SPT-K1...",B,4150.0,4150.0
63,Elektrik + Pre Series,A52YC1006,NaN,Main Circuit Cable,"95mmSQ , 600V, SpekTek: 307/SPT/K1003TB091/16",1393.0,meter,22734.0,22813.0,Main Circuit Cable,"95mmSQ , 600V, SpekTek: 307/SPT/K1003TB091/16",B,14379.0,14379.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5123,Jig Tools 6TS Pelokalan,B52TK0182,NaN,Cabel Ties,"T & B, Ty-Rap, TY 27M / Panduit BT4LH-L",0.0,Pcs,10000.0,10000.0,Cabel Ties,"T & B, Ty-Rap, TY 27M / Panduit BT4LH-L",B,140.2,140.2
5129,Jig Tools 6TS Pelokalan,D72SD10125,NaN,Tap,M10 X P 1.25,0.0,Pcs,30.0,30.0,TAP,M10 X P 1.25,B,25.0,25.0
5130,Jig Tools 6TS Pelokalan,D72SD12100,NaN,TAP SPIRAL,"M12 ; P:1,5 100",0.0,Pcs,30.0,30.0,TAP SPIRAL,"M12 ; P:1,5 100",B,23.0,23.0
5362,Jig Tool,A11RA0120X44,NaN,STEEL PLATE TEMBAGA,12 x 30 x 115,65.0,PCS,65.0,68.0,STEEL PLATE TEMBAGA,12 x 30 x 115,B,65.0,65.0


In [37]:
#data yang bisa dikatakan sudah PR
y = df_gabung[df_gabung["QTY PR TOTAL"] == df_gabung["Qty Requested"]]
y

,Nama Sheet Sumber,Kode Material,Kode Material Stock,Material / Komponen,Spesifikasi_x,Qty / TS (Series),UoM/Car,Qty Total (Series) + Allowance,QTY PR TOTAL,Mat. Description,Spesifikasi_y,PR Status,Qty Requested,Ordered
0,Elektrik + Pre Series,B336E12101,NaN,ELVP TC,Refer to dwg. 33.6-E12101 & 86.2-E12104 86.2-E...,2.0,set,32.0,32.0,ELVP TC,Refer to dwg. 33.6-E12101 <(>&<)> 86.2-E1210...,B,32.0,32.0
1,Elektrik + Pre Series,B336E12201,NaN,ELVP M1,Refer to dwg. 33.6-E12201 & 86.2-E12204 86.2-E...,3.0,set,48.0,48.0,ELVP M1,Refer to dwg. 33.6-E12201 <(>&<)> 86.2-E12201...,B,48.0,48.0
2,Elektrik + Pre Series,B336E12301,NaN,ELVP M2,Refer to dwg. 33.6-E12301 & 86.2-E12304 86.2-E...,3.0,set,48.0,48.0,ELVP M2,Refer to dwg. 33.6-E12301 <(>&<)> 86.2-E12301...,B,48.0,48.0
3,Elektrik + Pre Series,B336E12401,NaN,ELVP T1,"Refer to dwg. 33.6-E12401 & 86.2-E12404, 86.2...",2.0,set,32.0,32.0,ELVP T1,Refer to dwg. 33.6-E12401 <(>&<)> 86.2-E12401...,B,32.0,32.0
4,Elektrik + Pre Series,B336E12501,NaN,ELVP T2,Refer to dwg. 33.6-E12501 & 86.2-E12504 86.2-E...,1.0,set,16.0,16.0,ELVP T2,Refer to dwg. 33.6-E12501 <(>&<)> 86.2-E12501...,B,16.0,16.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6082,Tools & Consumable Tools Perbaikan,D62QF0609,NaN,Adhesive Remover,fas cleaner #5750 - adhesive remover,0.0,LITER,0.0,48.0,Adhesive Remover,Fas cleaner #5750 GR - Adhesive Remover,B,48.0,48.0
6098,Consumable Pre Series,D30WD0002,NaN,PASIR GRID,GRID BLASTING,0.0,Kg,0.0,0.0,PASIR GRID,GRID BLASTING,B,0.0,0.0
6099,Consumable Pre Series,D30WD0006,NaN,ALUMINIUM OXIDE,MESH 16 - GRIT F16 - BROWN ALUMUNIUM OXIDE - T...,0.0,kg,0.0,0.0,ALUMINIUM OXIDE,MESH NO. 24 - GRIT F24-BROWN FUSE ALUMINA (SE...,B,0.0,0.0
6124,Consumable Pre Series,D62QE0114,NaN,BITUMINOUS,SPEK.NO 001/TEK-SPEK/ALL PROYEK/14 REV. 0,0.0,Kg,0.0,0.0,BITUMINOUS,SPEK.NO 001/TEK-SPEK/ALL PROYEK/14 REV. 0,B,0.0,0.0


In [38]:
# Purchase Requisition chart

import plotly.express as px

pr = {
    'status PR': ['Belum PR', 'Sudah PR', 'Stock'],
    'Total': [len(x), len(y), len(instockData)],
}
df = pd.DataFrame(pr)

# 2. Buat pie chart
prchart = px.pie(df, values = 'Total', names = 'status PR', title = 'Purchase Requisition')
prchart

# Purchase Order Pie Chart

In [118]:
sapdataset_ringkas

,Kode Material,Mat. Description,Spesifikasi,Qty Requested,Ordered
0,16.09.2026,PEMADAM KEBAKARAN OTOMATIS,Satu set sistem pemadam kebakaran otomatis te...,1.0,0.0
1,561U03131,Flexible hose from UC to MRP,DRAWING 56.1-U03131,124.0,124.0
2,A007A18507,Piping for cooling genset,"Sesuai BOM Raw Material Sistem Mekanik, 205-B...",2.0,0.0
3,A007A18508,Piping for Drainase,"Sesuai BOM Raw Material Sistem Mekanik, 205-B...",2.0,0.0
4,A01LK0020,Kawat Tali,GALVANIS Dia. 2mm,2677.0,2061.0
...,...,...,...,...,...
12261,E97SC2000,Printer Marking Tube dan Heatshrink,SUPVAN TP2000M (spek dan gambar terlampir),1.0,1.0
12262,E97SH0002,ISO Container 20 feet,ISO 668,1.0,1.0
12263,E97SH0003,ISO Container 40 feet,ISO 668,1.0,1.0
12264,E97TC1321,Rubber Hardness,Shore Durometer HH-336 Shore A Digital/Compact,1.0,1.0


In [172]:
# N = Material masih belum digunakan
# A/K = Material dalam tahap ...?
# B = Material sudah dilakukan PO
statusN = sapdataset_ringkas[sapdataset_ringkas['PR Status'] == 'N']
statusAorK = sapdataset_ringkas[(sapdataset_ringkas['PR Status'] == 'A') | (sapdataset_ringkas['PR Status'] == 'K')]
statusB = sapdataset_ringkas[sapdataset_ringkas['PR Status'] == 'B']



po = {
    'Status PO': ['N', 'A/K', 'B'],
    'Total': [len(statusN), len(statusAorK), len(statusB)],
}
df = pd.DataFrame(po)

# 2. Buat pie chart
pochart = px.pie(df, values = 'Total', names = 'Status PO', title = 'Purchase Order')
pochart

# Pembagian data

In [138]:
bomdataset2 = pd.read_csv(bomurl, low_memory=False)
sapdataset2 = pd.read_csv(sapdataseturl, low_memory=False)

# satu kode material, banyak kali PR.
# banyak kode material, satu kali PR.
# satu kode material, banyak kali PR (ada kode material yang menggunakan stock, jadi tidak PR)
# banyak kode material, satu kali PR (ada kode material yang menggunakan stock, jadi tidak PR)

In [156]:
#Pengelompokkan data material yang tidak memiliki dan memiliki stock

tidakadastockdata = bomdataset2[bomdataset2["Kode Material Stock"].isna()]
adastockdata = bomdataset2[bomdataset2["Kode Material Stock"].notna()]

In [163]:
noStockGroupData = tidakadastockdata.groupby("Kode Material").size()
noStockGroupData

,0
Kode Material,
A01LK0020,1
A02YC032150,1
A04TH0085,1
A09SI015D35,3
A09SI050D35,1
...,...
E97BE2441,1
E97BE2442,1
E97BE2444,1


In [179]:
noStockGroupData = noStockGroupData.to_frame(name="Jumlah")
noStockGroupData[noStockGroupData['Jumlah'] > 3]

,Jumlah
Kode Material,
A09SK0001,4
A09TA0050,4
A10PW0350D7,6
A10TH004007,6
A10TH2020,4
...,...
D78UA0050E,7
D82WI0065,5
D83UX0000B-KEN,6


In [180]:
bomdataset2[bomdataset2['Kode Material'] == 'D78UA0050E']

,Nama Sheet Sumber,Kode Material,Kode Material Stock,Material / Komponen,Spesifikasi,Qty / TS (Series),UoM/Car,Qty Total (Series) + Allowance,QTY PR TOTAL
367,De-Scoping,D78UA0050E,NaN,NaN,NaN,60.0,BUAH,0.0,0.0
503,De-Scoping,D78UA0050E,NaN,NaN,NaN,24.0,BUAH,0.0,0.0
602,De-Scoping,D78UA0050E,NaN,NaN,NaN,4.1,PCS,41.0,0.0
5291,Consumable Tools,D78UA0050E,NaN,KUAS LURUS,"UK. 2""",49.2,PCS,492.0,499.0
5507,Consumable Series,D78UA0050E,NaN,NaN,"UK.2""",60.0,BUAH,0.0,0.0
5643,Consumable Series,D78UA0050E,NaN,NaN,"UK.2""",24.0,BUAH,0.0,0.0
6991,Consumable Pre Series,D78UA0050E,NaN,KUAS LURUS,"UK.2""",0.0,BUAH,0.0,0.0


In [181]:
sapdataset2[sapdataset2['Kode Material'] == 'D78UA0050E']['Qty Requested']

,Qty Requested
10326,154
20100,228
27282,20
42275,1
42429,30
42484,30
42609,345
43706,15
44422,309
47398,38


In [182]:
komat = "D78UA0050E"

sapdataset['Qty Requested'] = pd.to_numeric(sapdataset['Qty Requested'], errors='coerce').fillna(0)
total_qty = sapdataset.loc[sapdataset['Kode Material'] == komat, 'Qty Requested'].sum()

print(f"Total Qty Requested untuk {komat} adalah:", total_qty)

Total Qty Requested untuk D78UA0050E adalah: 1285.0


In [39]:
sapdataset[]

,WBS Description,Kode Material,Mat. Description,Spesifikasi,Purch.Req.,PR Status,Req.Date,PR Deliv. Dat,Chngd on,Qty Requested,Un,PO Number,PO Date,PO Deliv. Dat,Ordered,Created by,Requested By,Matl Group
0,PENGADAAN SARANA 186 CAR LRT JABODEBEK,A10MZ0030,Plate PA66,t3x1000x2000,10009794.0,B,06.03.2018,30.07.2018,04.07.2018,8.000,SHT,4500004709,20.09.2018,12.10.2018,8.000,INKA_REN_PR2,ANdA,RMRAWM
1,PENGADAAN SARANA 186 CAR LRT JABODEBEK,A15ND0300,Round Bar,"SUS 304, Ø30x6000",10009794.0,B,06.03.2018,24.08.2018,04.07.2018,2.000,STK,4500003602,09.04.2018,16.07.2018,2.000,INKA_REN_PR2,ANdA,RMRAWM
2,PENGADAAN SARANA 186 CAR LRT JABODEBEK,A15ST0300,ROUND BAR,S355J2+N Æ30x6000,10009794.0,B,06.03.2018,30.07.2018,04.07.2018,6.000,STK,4500004037,11.06.2018,24.08.2018,6.000,INKA_REN_PR2,ANdA,RMRAWM
3,PENGADAAN SARANA 186 CAR LRT JABODEBEK,A15ST0450,ROUND BAR,S355J2+N Æ45x6000,10009794.0,B,06.03.2018,30.07.2018,04.07.2018,51.000,STK,4500004037,11.06.2018,24.08.2018,51.000,INKA_REN_PR2,ANdA,RMRAWM
4,PENGADAAN SARANA 186 CAR LRT JABODEBEK,B44LA0016N,WASHER P. ( NORD LOCK ) 4T M16 BAJA 8.8T,M 16.0 BAJA 8.8T ; terlampir,10011014.0,B,24.09.2018,29.10.2018,24.09.2018,56.544,PC,4500005117,14.11.2018,04.01.2019,56.544,INKA_REN_PR1,mujiyo,RMFAST
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108295,PENGADAAN 16 TS KRL KCI,D83SPB001#HT,SPIDOL BESAR,HITAM (BOARDMARKER),10037487.0,N,16.09.2026,23.09.2026,16.09.2026,5.000,PC,NaN,00.00.0000,00.00.0000,0.000,PPC_RENP05,RO-MEMO,RMCONS
108296,PENGADAAN 16 TS KRL KCI,D66UG0024-NCI,ISOLASI KERTAS,UKURAN 24 mm,10037487.0,N,16.09.2026,23.09.2026,16.09.2026,5.000,ROL,NaN,00.00.0000,00.00.0000,0.000,PPC_RENP05,RO-MEMO,RMCONS
108297,PENGADAAN 16 TS KRL KCI,D31WH5921,MOLYKOTEPASTE,ENG-N PLUS - D COMING,10037487.0,N,16.09.2026,23.09.2026,16.09.2026,1.000,KG,NaN,00.00.0000,00.00.0000,0.000,PPC_RENP05,RO-MEMO,RMCONS
108298,PENGADAAN 16 TS KRL KCI,D68WH0001,GREASE,GREASE PERTAMINA SGX NL 1 PAIL @16 KG,10037487.0,N,16.09.2026,23.09.2026,16.09.2026,2.000,PAI,NaN,00.00.0000,00.00.0000,0.000,PPC_RENP05,RO-MEMO,RMCONS


In [40]:
x

,Nama Sheet Sumber,Kode Material,Kode Material Stock,Material / Komponen,Spesifikasi_x,Qty / TS (Series),UoM/Car,Qty Total (Series) + Allowance,QTY PR TOTAL,Mat. Description,Spesifikasi_y,PR Status,Qty Requested,Ordered
6,Elektrik + Pre Series,B336E12003,NaN,TMJB,Refer to dwg. 33.6-E12003 33.2-E12004 & 86.2-E...,12.0,set,192.0,192.0,TMJB,Refer to dwg. 33.2-E12004 <(>&<)> 86.2-E12003...,B,216.000,216.000
23,Elektrik + Pre Series,B52UM1602,NaN,Fibox,"Fibox, ALN081306,127 X 81 X 57 MM,ALUMINIUM",2.0,un,32.0,0.0,Fibox,"Fibox, ALN081306,127 X 81 X 57 MM,ALUMINIUM",B,204.222,204.222
37,Elektrik + Pre Series,B52TI0176,NaN,"PUSH BUTTON, APT LA39- C1-20D/R26","PUSH BUTTON, APT LA39-C1-20D/R26-JW-RW",2.0,Pcs,32.0,32.0,"PUSH BUTTON, APT LA39-C1-20D/R26","PUSH BUTTON, APT LA39-C1-20D/R26-JW-RW",B,36.000,36.000
38,Elektrik + Pre Series,B52TI0177,NaN,"PUSH BUTTON, APT LA39- C1-20D/G26","PUSH BUTTON, APT LA39-C1-20D/G26-JW-R",4.0,Pcs,64.0,64.0,"PUSH BUTTON, APT LA39-C1-20D/G26","PUSH BUTTON, APT LA39-C1-20D/G26-JW-RW",B,76.000,76.000
39,Elektrik + Pre Series,B52TI2203,NaN,X-COM Male Connector,"Wago, X-COM Male Plugs 769-605",6.0,Pcs,96.0,96.0,X-COM Male Connector,"Wago, X-COM Male Plugs 769-605",B,529.000,529.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6164,Consumable Pre Series,D50EB651032AX#RD,NaN,STICKER ORACAL RED,UK 1260 X 50000mm (651G-032),0.0,Roll,0.0,0.0,STICKER ORACAL LIGHT RED,UK 1260 X 50000mm (651G-032),B,150.000,82.000
6165,Consumable Pre Series,D50EB651010AX#WT,NaN,STICKER ORACAL WHITE,UK 1260 X 50000mm (651G-010),0.0,Roll,0.0,0.0,STICKER ORACAL WHITE,UK 1260 X 50000mm (651G-010),B,47.000,47.000
6166,Consumable Pre Series,D68QH0713577,NaN,PIPE SEALANT,MXLOC 13577 PIPE SEALANT,0.0,mL,0.0,100.0,PIPE SEALANT,MXLOC 13577 (1Pc=50mL),B,1903.000,1903.000
6167,Consumable Pre Series,D68QH0700577,NaN,TRHEAD LOCKER,MXLOC 577 THREAD LOCKER ADHESIVE,0.0,mL,0.0,100.0,THREAD LOCKER ADHESHIVE,MXLOC 577 (1Pc=50Ml),B,1903.000,1903.000
